In [ ]:
import kagglehub
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
# Task: Inspect first few rows
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Task: Handle missing values

# First, let's see which columns actually have holes in them.
print("Missing values before fix:")
print(df.isnull().sum())

# Now, I'm filling the missing values.
# Since I don't know what 'P_2' or 'B_1' actually mean so i can't guess the value
# For numbers: I use median because financial data usually has big outliers
# which would mess up the mean. Median is safer.
# For text/categories: I use mode (the most common value).

for col in df.columns:
    if df[col].dtype == 'object':
        # If it's text (categorical), fill with the most common class
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        # If it's a number, fill with the median
        df[col] = df[col].fillna(df[col].median())

# Verification step
print("\nMissing values after fix:")
print(df.isnull().sum())

In [ ]:
# Task 2: Write your code here:
# Check and remove duplicates

# 1. Check count
duplicate_count = df.duplicated().sum()
print(f"Number of duplicates found: {duplicate_count}")

# 2. Remove duplicates (if any)
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates to remove.")

# 3. Final Check
print(f"Final Data Shape: {df.shape}")

In [ ]:
# Task 3: Write your code here:
# Encode categorical variables

# First, I need to check if there are any text columns (Type 'object').
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(categorical_cols)}")

if len(categorical_cols) > 0:
    # If there are text columns, I use One Hot Encoding (get_dummies).
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
    print("Encoding complete.")
else:
    print("No categorical variables found. Skipping encoding.")

# Showing the new shape of the dataframe to see how many columns were added
print(f"New Data Shape: {df.shape}")
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

target_col = df.columns[-1]
print(f"I found the target column name: '{target_col}'")


# 1. Separate Features (X) and Target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. Initialize Scaler
scaler = StandardScaler()

# 3. Fit and Transform
X_scaled_array = scaler.fit_transform(X)

# 4. Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print("Scaling complete.")
X_scaled.head()

In [ ]:
# Task 5: Write your code here:
# Task: Check for target imbalance

# 1. Identify the target column
target_col = df.columns[-1]
print(f"Target column identified as: '{target_col}'")

# 2. Count the values
class_counts = df[target_col].value_counts()
class_percentages = df[target_col].value_counts(normalize=True) * 100

print("\nClass Counts:")
print(class_counts)

print("\nClass Percentages:")
print(class_percentages)

# 3. State if Imbalanced
minority_class_percent = class_percentages.min()

if minority_class_percent < 20:
    print(f"\nCONCLUSION: The dataset is IMBALANCED (Minority class is only {minority_class_percent:.2f}%).")
else:
    print(f"\nCONCLUSION: The dataset is BALANCED.")

In [ ]:
# Task 1: Write your code here:
# Task: Split the dataset into features (X) and target (y)

# 1. Define Features (X)
X = X_scaled

# 2. Define Target (y)
target_col = df.columns[-1]
y = df[target_col]

# 3. Verification
print("Data split into X and y.")
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

# Step 1: Defining the Split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 2: Defining the Model
model = CatBoostClassifier(random_state=42, verbose=0)

f1_scores = []
acc_scores = []

print("Starting 5-Fold Cross-Validation...")

# Step 3: Training Loop

# I'm looping through 5 different splits.
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):

    # Splitting the data
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Training
    model.fit(X_train, y_train)

    # Predicting
    y_pred = model.predict(X_test)

    # Calculating Scores
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    f1_scores.append(f1)
    acc_scores.append(acc)

    print(f"Fold {fold+1} | F1 Score: {f1:.4f} | Accuracy: {acc:.4f}")

# --- Step 4: Final Output ---

average_f1 = np.mean(f1_scores)
average_acc = np.mean(acc_scores)

print("-" * 30)
print(f"Average F1 Score:   {average_f1:.4f}")
print(f"Average Accuracy:   {average_acc:.4f}")

In [ ]:
# Task 1: Write your code here:


#Plot feature importance and find the Golden Feature
feature_importances = pd.Series(model.feature_importances_, index=X.columns)

# 2. Find the "Golden Feature"
# I'm simply looking for the maximum value in the series.
golden_feature_name = feature_importances.idxmax()
golden_feature_score = feature_importances.max()

print(f"🌟 THE GOLDEN FEATURE IS: '{golden_feature_name}'")
print(f"Importance Score: {golden_feature_score:.2f}")

# 3. Plotting
plt.figure(figsize=(10, 6))

feature_importances.nlargest(10).sort_values().plot(kind='barh', color='gold')

plt.title('Top 10 Features (Golden Feature Highlighted)')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Task Bonus: Write your code here: